In [1]:
import serial
import time
import logging
import coloredlogs

coloredlogs.install(level=logging.ERROR)

# Initialize serial port
RADAR_DEV = '/dev/tty.usbserial-5'

In [2]:
#ser = serial.Serial(RADAR_DEV, 256000, timeout=5) 
#print(ser)

from LD2410 import *
import time
radar = LD2410(port=RADAR_DEV) # Replace <port> with your serial port. e.g "COM3" or "/dev/ttyS0" etc. 

In [3]:
radar.enable_engineering_mode() # Enable engineering mode to get full data
radar.start() # Start the radar module

In [4]:
from IPython.core.display import display, HTML
from IPython.display import clear_output
from IPython.display import display, Javascript
from IPython.display import display, HTML, update_display

def scroll_to_bottom():
    display(Javascript("window.scrollTo(0, document.body.scrollHeight);"))

def get_color(val):
    c = int(((100-val)/100.0) * 250)
    return f'ff{c:02x}{c:02x}'

def get_html_metric(val):
    return f'<span style="color: #{get_color(val)}; font-family: monospace; font-size: 24px; display: inline-block; width: 4ch; overflow: hidden; text-overflow: ellipsis;">{val:3d}</span>'
    
def render_gates(gates):
    return " , ".join([get_html_metric(val) for val in gates]) 

def render_out(data):
    return render_gates(data[1]) + ' <span style="margin-right: 250px;"></span> ' + render_gates(data[2])

display(HTML(render_out(([3, 88, 100, 75, 100, 69], [8, 62, 100, 100, 28, 8, 5, 10, 15, 8], [0, 0, 100, 100, 26, 22, 17, 19, 12, 92]))))
# for x in range(10):
#     display(HTML(render_out(([3, 88, 100, 75, 100, 69], [8, 62, 100, 100, 28, 8, 5, 10, 15, 8], [0, 0, 100, 100, 26, 22, 17, 19, 12, 92]))))
# display(HTML(render_gates([8, 43, 100, 90, 31, 4, 11, 2, 10, 10])))
# for x in range(1,100,10):
# Create the HTML string and display it
#     display(HTML(get_html_metric(x)))
#display(HTML(f'<p style="color: #ff3333;">{text}</p>'))


/var/folders/44/r41gz5s10vs3dk_rt7cpdt1m0000gn/T/ipykernel_99315/899089453.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [ ]:
import asyncio
from collections import deque

out_buffer = deque(maxlen=25)
display_obj = display(HTML("<div style='font-family:Courier New, monospace;'>Starting...</div>"), display_id=True)


async def async_loop():
# def async_loop():
    while True:
        dt = radar.get_data()
        out_buffer.append(dt)
        
        htmlc = "\n".join([f"<p>{render_out(val)}</p>" for val in out_buffer])
        html = HTML(htmlc)
        
        display_obj.update(html)
        await asyncio.sleep(0.25)

await async_loop()  # Run the asynchronous loop
# async_loop()  # Run the asynchronous loop

In [ ]:
print(radar.last_detection)
print(radar.bt_query_mac())
print(radar.read_detection_params())
print(radar.read_firmware_version())

In [ ]:
radar.bt_enable()

In [ ]:
class Target:
    def __init__(self, distance, energy):
        self._distance = distance
        self._energy = energy


class MovingTarget(Target):
    def __init__(self, distance, energy):
        Target.__init__(self, distance, energy)
    
    def __repr__(self) -> str:
        return f"Moving Target at {self._distance}cm with energy {self._energy}"

class StationnaryTarget(Target):
    def __init__(self, distance, energy):
        Target.__init__(self, distance, energy)
    
    def __repr__(self) -> str:
        return f"Stationnary Target at {self._distance}cm with energy {self._energy}"


class TargetFactory:


    def get(self, buffer):
        targets = []
        target_type_ = buffer[8]
        stationary_target_distance_ = buffer[9] + (buffer[10] << 8)
        stationary_target_energy_ = buffer[14]
        moving_target_energy_ = buffer[11]
        moving_target_distance_ = buffer[15] + (buffer[16] << 8)
        if target_type_ == 0x01:
            targets.append(StationnaryTarget(stationary_target_distance_, stationary_target_energy_))
        elif target_type_ == 0x02:
            targets.append(MovingTarget(moving_target_distance_, moving_target_energy_))
        elif target_type_ == 0x03:
            targets.append(StationnaryTarget(stationary_target_distance_, stationary_target_energy_))
            targets.append(MovingTarget(moving_target_distance_, moving_target_energy_))
        return targets


class LD2410:

    def __init__(self, port):
        self._port = serial.Serial(port=port,baudrate=256000)
        self._buffer = bytearray()
        self._factory = TargetFactory()

    def read(self):
        if self._port.in_waiting > 0:
            new_data = self._port.read(self._port.in_waiting)
            self._buffer += new_data
        while len(self._buffer) > 0 and self._buffer[0] != 0xF4:
            self._buffer = self._buffer[1:]
        if len(self._buffer) > 0 and self._buffer[0] == 0xF4:
            if len(self._buffer) >= 23:
                targets = []
                start_pattern = bytearray([0xF4, 0xF3, 0xF2, 0xF1, 13, 0x00, 0x02, 0xAA])
                end_pattern = bytearray([0x55, 0x00, 0xF8, 0xF7, 0xF6, 0xF5])
                if start_pattern == self._buffer[0:len(start_pattern)] and end_pattern == self._buffer[17:]:
                    targets = self._factory.get(self._buffer)
                self._buffer = self._buffer[23:]
                return targets
        return None


In [ ]:
radar = LD2410(RADAR_DEV)
while True:
    targets = radar.read()
    if targets:
        print(targets)

In [ ]:
import logging
logging.basicConfig(level=logging.DEBUG)
import serial
from threading import Thread
from collections import deque
from datetime import datetime,timedelta
import time
import random



class Thresholds:
    def __init__(self,
                moving:list[int|None]|None,
                static:list[int|None]|None):
        """
                None or missing parameters indicates no threshold set at this range 
                Whole lists can be ommitted if no thresholds of that type are required
        """
        self.moving=moving
        self.static=static


class Reading:
    TARGET_STATES={0x00:"No target",
                    0x01:"Moving target",
                    0x02:"Static target",
                    0x03:"Moving & static target"}

    def __init__(self,
                    target_state:int,
                    moving_target_range:int,
                    moving_target_energy:int,
                    static_target_range:int,
                    static_target_energy:int,
                    detection_distance:int,
                    max_moving_distance:int,
                    max_static_distance:int,
                    moving:list[int],
                    static:list[int],
                    timestamp:datetime|None=None):
        """
            Stores a single reading from the sensor
        """
        if timestamp is None:
            timestamp=datetime.now()
        self.timestamp=timestamp
        self.moving:list[int]=moving
        self.static:list[int]=static
        self.target_state:int=target_state
        self.moving_target_range:int=moving_target_range
        self.moving_target_energy:int=moving_target_energy
        self.static_target_range:int=static_target_range
        self.static_target_energy:int=static_target_energy
        self.detection_distance:int=detection_distance
        self.max_moving_distance:int=max_moving_distance
        self.max_static_distance:int=max_static_distance

   

    def __str__(self)->str:
        return f"{self.timestamp}\tMoving: {self.moving}\tStatic: {self.static}"


    def __repr__(self)->str:
        return str(self)


    def compare(self,thresholds:Thresholds)->bool:
        """
            Tests if levels in this reading exceed the Thresholds
        """
        return False # Placeholder

    @classmethod
    def new_from_bytes(cls,seq:bytes)->object:
        movs:list[int]=[seq[11+i] for i in range(12)]
        stats:list[int]=[seq[23+i] for i in range(12)]
        res=cls(target_state=seq[0],
                    moving_target_range=byte_pair_to_int(seq[1:3]),
                    moving_target_energy=seq[3],
                    static_target_range=byte_pair_to_int(seq[4:6]),
                    static_target_energy=seq[6],
                    detection_distance=byte_pair_to_int(seq[7:9]),
                    max_moving_distance=seq[9],
                    max_static_distance=seq[10],
                    moving=movs,
                    static=stats
                )
        return res

        



class LD2410Exception(Exception):
    pass

def list_ints_to_bytes(inp:list[int])->bytes:
    res=b''
    for b in inp:
        res+=b.to_bytes(1,byteorder="little",signed=False)
    return res

def bytes_to_list_ints(inp:bytes)->list[int]:
    bits=[c for c in inp]
    return bits

def byte_pair_to_int(inp:bytes)->int:
    return int.from_bytes(inp,byteorder="little",signed=False)

def bytes_to_hex(inp:bytes)->str:
    digits:list[int]=bytes_to_list_ints(inp)
    items=[f"0x{i:02x}" for i in digits]
    return ",".join(items)

class LD2410(Thread):
    COMMAND_PREAMBLE:bytes=list_ints_to_bytes([0xFD,0xFC,0xFB,0xFA])
    COMMAND_POSTAMBLE:bytes=list_ints_to_bytes([0x04,0x03,0x02,0x01])
    REPORTING_PREAMBLE:bytes=list_ints_to_bytes([0xF4,0xF3,0xF2,0xF1])
    REPORTING_POSTAMBLE:bytes=list_ints_to_bytes([0xF8,0xF7,0xF6,0xF5])



    def __init__(self,
                    port:str,
                    baud:int=256000,
                    maxlen:int=50,
                    thresholds:Thresholds=None):

        """

            maxlen is the maximum number of measurements kept in the buffer (deque)
            thresholds is a Threshold object for testing prescence


            .set_thresholds
            .get_latest
            .is_present
            
        
        """
        self.port=port
        self.baud=baud
        self.thresholds:Thresholds=thresholds # Default is no threshold set
        self.running:bool=False
        self.stopped:bool=True
        self.buffer:deque=deque(maxlen=maxlen)
        self.serial:serial.Serial|None=None

        
        

        Thread.__init__(self) # Complete thread initialisation
        self.daemon=True

    def read_bytes(self,length:int)->bytes:
        r:bytes=self.serial.read(length)
        #logging.debug(f"\t\t>>>> {bytes_to_hex(r)}")
        return r



    def _tx_sequence(self,sequence:bytes|list[int]):

        # Convert a list of ints to a bytes obj if required
        if isinstance(sequence,bytes):
            payload:bytes=sequence
        else:
            payload:bytes=list_ints_to_bytes(sequence)

        #Preamble
        self.serial.write(self.COMMAND_PREAMBLE)
        #Length
        length:int=len(payload)
        length_b:bytes=length.to_bytes(2,byteorder="little")
        self.serial.write(length_b)
        #Payload
        self.serial.write(payload)
        #Postamble
        self.serial.write(self.COMMAND_POSTAMBLE)
        


    def _read_any_message(self):
        """
                Just reads the next message, returns a tuple
                (preamble:bytes,message:bytes,postamble:bytes
                does not check anything except for timeouts
        """
        # Wait until we have the first byte of a preamble
        #logging.debug("Fetching a response")
        preamble_target_no:int=0 #
        preamble:bytes=b"" 
        target_preamble:bytes=b'' # set according to the first byte spotted matching preamble

        while True:
            #print(".",end="")
            c_b:bytes=self.serial.read(1)
            #print("%",end="")
            if len(c_b)==0:
                logging.debug("timeout!")
            else:
                c:int=c_b[0]
                #print(f"!0x{c:02x},",end="")
                if preamble_target_no==0:
                    if (c==self.COMMAND_PREAMBLE[0] or c==self.REPORTING_PREAMBLE[0]):
                        # We found the first byte of preamble (maybe!)

                        # Figure out what to expect next
                        if c==self.REPORTING_PREAMBLE[0]: # First char sets the type
                            target_preamble=self.REPORTING_PREAMBLE
                        else:
                            target_preamble=self.COMMAND_PREAMBLE

                        # Move to next one as target!
                        preamble+=bytes([c])
                        preamble_target_no+=1
                        #print("\nFound first!\n")
                else:
                    # Must be second character here!
                    if not c==target_preamble[preamble_target_no]:
                        # Bad subsequent char, so restart
                        preamble=b""
                        target_preamble=b""
                        preamble_target_no=0
                        #print("Sequence broken\n\n")
                    else:
                        # We found the next good character
                        preamble_target_no+=1
                        #print(f"\nFound {preamble_target_no}th character in preamble")
                        preamble+=bytes([c])
                        if preamble_target_no==4:
                            # We finished finding the preamble
                            #print(f"\nPreamble Done OK")
                            break


        length_raw:bytes=self.read_bytes(2)
        if not len(length_raw)==2:
            raise LD2410Exception("Didn't get two bytes for the message length in the expected time")
        length:int=byte_pair_to_int(length_raw)
        #logging.debug(f"\t\t\t\tPayload detected is {length} bytes")


        payload:bytes=self.read_bytes(length)
        if not len(payload)==length:
            raise LD2410Exception(f"Payload was short, only received : {len(payload)} bytes")
        #logging.debug(f"\t\t\tPayload received: {bytes_to_hex(payload)}")


        #Read and check postamble
        postamble:bytes=self.read_bytes(4)
        if not len(postamble)==4:
            raise LD2410Exception(f"Failed to receive the full 4 byte postamble")
        #logging.debug("\t\t\t\t\tpostamble OK")
        return (preamble,payload,postamble)
        
    def _flush_serial(self):
        self.serial.reset_input_buffer()


    def _rx_sequence(self,report_expected:bool,
                        retries=10,
                        timeout=1)->bytes:
        """
            Fetches a response from the sensor

                report_expected
                         is True if we are waiting for a measurement report
                         or False if we want a command ack or response
                retries
                         is the number of times it will try again to 
                         get the desired type of response

                timeout
                        is the serial port timeout used
        
        """
        
        if report_expected:
            expected_preamble=self.REPORTING_PREAMBLE
            expected_postamble=self.REPORTING_POSTAMBLE
        else:
            expected_preamble=self.COMMAND_PREAMBLE
            expected_postamble=self.COMMAND_POSTAMBLE
        if self.serial is not None:
            self.serial.timeout=timeout
        
        for try_no in range(retries):
            preamble,payload,postamble=self._read_any_message()
            if preamble==expected_preamble and postamble==expected_postamble:
                return payload
            logging.debug("Ignoring a message with the wrong pre-post-ambles")

        raise LD2410Exception(f"Failed to get a message of type {'report' if report_expected else 'command ack'} after {retries} attempts")

    def _enter_configuration_mode(self):
        logging.debug("\n\n\n*********** Entering config mode")
        self._tx_sequence([0xFF,0x00,0x01,0x00])
        resp:bytes=self._rx_sequence(report_expected=False,retries=10,timeout=2)
        if not resp==list_ints_to_bytes([0xFF,0x01,0x00,0x00,0x01,0x00,0x40,0x00]):
            raise LD2410Exception("Failed to get ack to entering config mode")
        logging.debug("Successfully entered config mode")

    def _leave_configuration_mode(self):
        logging.debug("\n\n\n*********** Leaving configuration mode")
        self._tx_sequence([0xFE,0x00])
        resp:bytes=self._rx_sequence(report_expected=False,retries=5,timeout=2)
        if not resp==list_ints_to_bytes([0xFE,0x01,0x00,0x00]):
            raise LD2410Exception("Failed to get ack to leaving config mode")
        logging.debug("Successfully left config mode")


    def _request_firmware_version(self):
        
        logging.info("\n\n\n************ Requesting firmware version")
        self._tx_sequence([0xA0,0x00])
        resp:bytes=self._rx_sequence(report_expected=False,retries=3,timeout=2)
        logging.info(f"Firmare code found: {bytes_to_hex(resp)}")
        


    def _send_reset(self):
        logging.info("\n\n\n************ Sending reset")
        self._tx_sequence([0xA3,0x00])
        resp:bytes=self._rx_sequence(report_expected=False,retries=3,timeout=2)
        if not resp==list_ints_to_bytes([0xA3,0x01,0x00,0x00]):
            raise LD2410Exception("Failed to get an ack on the attempt to reset")
        logging.debug("Reset acknowledged")


    def _start_engineering_mode(self):
        logging.info("\n\n\n************ Entering engineering mode")
        self._tx_sequence([0x62,0x00])
        resp:bytes=self._rx_sequence(report_expected=False,retries=10,timeout=2)
        if not resp==list_ints_to_bytes([0x62,0x01,0x00,0x00]):
            raise LD2410Exception(f"Failed to get the correct ack for entering engineering mode, instead rx'ed : ")

        logging.debug("Successfully entered engineering mode (ack'd)")
        

    def __enter__(self):
        if not self.serial is None:
            raise serial.SerialException("Attempt to open a port via a context manager (with) that is already open")
        if not self.port == "test":
            self.serial=serial.Serial(port=self.port,
                                baudrate=self.baud,
                                bytesize=serial.EIGHTBITS,
                                stopbits=serial.STOPBITS_ONE)




            

        self.start() # Launch own thread

    


        return self


    def __exit__(self,exc_type, exc_val, exc_tb):
        # Stop thread
        if not self.running:
            raise LD2410Exception("Attempt to properly end the monitoring thread, but somehow it's already gone!")

        self.running=False
        timeout:datetime=datetime.now()+timedelta(seconds=20)
        while True:
            if datetime.now()>timeout:
                raise LD2410Exception("Failed to stop the thread inside the timeout")
            if self.stopped: # Shut down elegantly within the timeout :-)
                break
            time.sleep(0.1)


        self.serial.close()

    def _get_next_reading(self,timeout_s:float=2)->Reading:
        """
            Grabs the next reading from the sensor, returns it 
            raises LD2410Exception if no reading obtained within the timeout
        """
        # self._enter_configuration_mode()
        # self._start_engineering_mode()
        while True:
            # Deal with test mode
            if self.port=="test":
                time.sleep(2)
                movs:list[int]=[random.randrange(1,50) for _ in range(9)]
                stat:list[int]=[random.randrange(1,50) for _ in range(9)]
                return Reading(moving=movs,static=stat)

            # Take a proper reading!
            #logging.debug("getting a new reading")
            seq:bytes=self._rx_sequence(report_expected=True,retries=1,timeout=2)
            #logging.debug(f"Received payload back of : {bytes_to_hex(seq)}")
            if len(seq)==35:
                break

        return Reading.new_from_bytes(seq)

        

    def set_presence_thresholds(self,levels:Thresholds):
        self.thresholds=levels

    def _test_presence(self,reading:Reading)->bool:
        """
            tests if the latest reading indicates presence 
            when compared the the thresholds
        """
        return reading.compare(self.thresholds)

    def get_latest(self)->Reading:
        while len(self.buffer)==0:
            logging.debug("Waiting for buffer to be refilled")
            time.sleep(0.1)
        return self.buffer[-1]

    def run(self):
        self.running=True
        self.stopped=False
        logging.debug("Thread running")
                # # Just to test
        # temp:bytes=self.serial.read(1024)
        # print(bytes_to_hex(temp))
        # stop
        self._enter_configuration_mode()
        self._send_reset()
        logging.debug("brief pause")
        time.sleep(2)
        self._flush_serial()
        self._enter_configuration_mode()
        self._request_firmware_version()
        #self._leave_configuration_mode()

        self._start_engineering_mode()
        self._leave_configuration_mode()

        while (self.running):
            """
                    Main loop that gets readings back from the sensor
            """
            read:Reading=self._get_next_reading()
            #logging.debug("NEW READING IS RECORDED")
            self.buffer.append(read)
            logging.info(read)
            #logging.debug("added to buffer ¬¬¬¬¬¬¬¬")
            if self.thresholds is not None:
                self.present = self._test_presence(read)
                logging.info(self.present)
    
        self.stopped=True


In [ ]:
def test_sensor():
    with LD2410(RADAR_DEV,maxlen=20) as sensor:
        time.sleep(0)
        for _ in range(100):
            time.sleep(0.5)
            print(f"\n\nREADING SET\n{sensor.get_latest()}")
test_sensor()

In [ ]:
with LD2410(RADAR_DEV,maxlen=20) as sensor:
    sensor.run()

In [ ]:
import utime
import machine
SPG30_SCL_PIN = 22
SPG30_SDA_PIN = 21
 
i2c = machine.SoftI2C(scl=machine.Pin(SPG30_SCL_PIN), sda=machine.Pin(SPG30_SDA_PIN))
i2c.start()
 
from ph4_esp32.sensors.scd4x import SCD4X
 
scd4x = SCD4X(i2c)
scd4x.start_periodic_measurement()

while True:
    utime.sleep_ms(1000)
    if scd4x.data_ready:
        co = scd4x.CO2
        tmp = scd4x.temperature
        humd = scd4x.relative_humidity
        print(f"Co: {co}, temp: {tmp}, humd: {humd}")

In [2]:
import serial, struct, time
import subprocess
from operator import invert
import os
import json
import socket
import paho.mqtt.publish as publish


def time_(): return int(time.time())


def datetime_():
    return time.strftime('%x %X', time.localtime())


class SPS30:
    NAME = 'SPS30'
    WARMUP = 20 # seconds
    
    def __init__(self, port, save_data=True, push_mqtt=False, INTERVAL=60):
        self.port = port
        self.interval = INTERVAL
        self.warmup = SPS30.WARMUP
        self.save_data = save_data
        self.push_mqtt = push_mqtt
        self.name = SPS30.NAME
        self.lastSample = 0
        self.fanOn = 0
        self.is_started = False
        self.ser = serial.Serial(self.port, baudrate=115200, stopbits=1, parity="N",  timeout=2)

    def __str__(self):
        return f'{self.port}, {self.name}, {self.fanOn}, {self.lastSample}'

    
    def start(self):
        self.ser.write([0x7E, 0x00, 0x00, 0x02, 0x01, 0x03, 0xF9, 0x7E])
        
    def stop(self):
        self.ser.write([0x7E, 0x00, 0x01, 0x00, 0xFE, 0x7E])
    
    def read_values(self):
        self.ser.flushInput()
        # Ask for data
        self.ser.write([0x7E, 0x00, 0x03, 0x00, 0xFC, 0x7E])
        toRead = self.ser.inWaiting()
        # Wait for full response
        # (may be changed for looking for the stop byte 0x7E)
        while toRead < 47:

            toRead = self.ser.inWaiting()
            print(f'Wait: {toRead}')
            time.sleep(1)
        raw = self.ser.read(toRead)
        
        # Reverse byte-stuffing
        if b'\x7D\x5E' in raw:
            raw = raw.replace(b'\x7D\x5E', b'\x7E')
        if b'\x7D\x5D' in raw:
            raw = raw.replace(b'\x7D\x5D', b'\x7D')
        if b'\x7D\x31' in raw:
            raw = raw.replace(b'\x7D\x31', b'\x11')
        if b'\x7D\x33' in raw:
            raw = raw.replace(b'\x7D\x33', b'\x13')
        
        # Discard header and tail
        rawData = raw[5:-2]
        
        try:
            data = struct.unpack(">ffffffffff", rawData)
        except struct.error:
            data = (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
        return data
    
    def read_serial_number(self):
        self.ser.flushInput()
        self.ser.write([0x7E, 0x00, 0xD0, 0x01, 0x03, 0x2B, 0x7E])
        toRead = self.ser.inWaiting()
        while toRead < 7:  #24
            toRead = self.ser.inWaiting()
            print(f'Wait: {toRead}')
            time.sleep(1)
        raw = self.ser.read(toRead)
        
        # Reverse byte-stuffing
        if b'\x7D\x5E' in raw:
            raw = raw.replace(b'\x7D\x5E', b'\x7E')
        if b'\x7D\x5D' in raw:
            raw = raw.replace(b'\x7D\x5D', b'\x7D')
        if b'\x7D\x31' in raw:
            raw = raw.replace(b'\x7D\x31', b'\x11')
        if b'\x7D\x33' in raw:
            raw = raw.replace(b'\x7D\x33', b'\x13')
        
        # Discard header, tail and decode
        serial_number = raw[5:-3].decode('ascii')
        return serial_number

    def run_query(self):
        if time_() - self.lastSample >= self.interval:
            if not self.is_started:   
                self.start()
                self.fanOn = time_()
                self.is_started = True
            if self.name == SPS30.NAME:
                name_  = self.read_serial_number()
                if len(name_) >0:
                    self.name = f'SPS_{name_}'
            if time_() - self.fanOn >= self.warmup:
                output = self.read_values()
                sensorData = ""
                for val in output:
                    sensorData += "{0:.2f},".format(val)

                output = ','.join([self.name, datetime_(),sensorData[:-1]])
                self.lastSample = time_()
                if self.is_started:
                    self.stop()
                    self.is_started = False
                if self.save_data:
                    print(output)
                    record_data(output)
                if self.push_mqtt:
                    push_mqtt_server(output)
            else:
                time.sleep(1)
        return None

    def close_port(self):
        self.ser.close()



In [6]:
p = SPS30(port='/dev/tty.usbserial-14140', push_mqtt=False, save_data=False)
# while True:
#     p.run_query()
#     time.sleep(10)

p.start()
print(p.read_serial_number())
while True:
    print(p.read_values())
    time.sleep(1)

Wait: 0
Wait: 7

Wait: 0
Wait: 47
(6.999496936798096, 7.420047283172607, 7.435161113739014, 7.442695140838623, 47.75303268432617, 55.60332489013672, 55.8591194152832, 55.87779235839844, 55.89006805419922, 0.5097849369049072)
Wait: 0
Wait: 48
(7.278012752532959, 7.713262557983398, 7.727295875549316, 7.734292984008789, 49.65830993652344, 57.81819534301758, 58.08210372924805, 58.101219177246094, 58.113956451416016, 0.5048573017120361)
Wait: 0
Wait: 47
(7.444939613342285, 7.887711048126221, 7.900044918060303, 7.906190395355225, 50.80348587036133, 59.14717102050781, 59.414642333984375, 59.43382263183594, 59.44683074951172, 0.5023009777069092)
Wait: 0
Wait: 48
(7.448545932769775, 7.8875861167907715, 7.896665573120117, 7.901200294494629, 50.83808135986328, 59.180416107177734, 59.44402313232422, 59.46261978149414, 59.4755973815918, 0.49453938007354736)
Wait: 0
Wait: 47
(7.471686363220215, 7.906688690185547, 7.911349296569824, 7.913682460784912, 51.00967788696289, 59.37055206298828, 59.62951278

Wait: 0
Wait: 47
(7.587316036224365, 8.027673721313477, 8.031270980834961, 8.033068656921387, 51.80257797241211, 60.290958404541016, 60.55253219604492, 60.5704460144043, 60.5836067199707, 0.5112177133560181)
Wait: 0
Wait: 48
(7.673863887786865, 8.121894836425781, 8.127720832824707, 8.130629539489746, 52.386775970458984, 60.97560501098633, 61.24285125732422, 61.2613639831543, 61.27470016479492, 0.5120135545730591)
Wait: 0
Wait: 48
(7.743790149688721, 8.198905944824219, 8.207254409790039, 8.211419105529785, 52.85654830932617, 61.52773666381836, 61.80045700073242, 61.81959533691406, 61.833072662353516, 0.5069247484207153)
Wait: 0
Wait: 47
(7.77982759475708, 8.238982200622559, 8.248956680297852, 8.253925323486328, 53.09767532348633, 61.81184387207031, 62.087772369384766, 62.10728454589844, 62.120845794677734, 0.5048246383666992)
Wait: 0
Wait: 48
(7.753021240234375, 8.211356163024902, 8.221925735473633, 8.22719669342041, 52.912784576416016, 61.597965240478516, 61.873714447021484, 61.8932762

Wait: 0
Wait: 47
(6.772148609161377, 7.19273042678833, 7.218635559082031, 7.231537342071533, 46.1673469543457, 53.781349182128906, 54.042694091796875, 54.06281661987305, 54.0748176574707, 0.5281225442886353)
Wait: 0
Wait: 48
(6.889837265014648, 7.3185505867004395, 7.345585823059082, 7.359050273895264, 46.96757888793945, 54.71502685546875, 54.98173904418945, 55.00233840942383, 55.01455307006836, 0.5287889242172241)
Wait: 0
Wait: 47
(6.934565544128418, 7.366710662841797, 7.394459247589111, 7.408272743225098, 47.270851135253906, 55.069480895996094, 55.338584899902344, 55.359413146972656, 55.37171173095703, 0.5269439220428467)
Wait: 0
Wait: 48
(6.972848892211914, 7.408490180969238, 7.437302589416504, 7.451651096343994, 47.52900695800781, 55.37220764160156, 55.6439208984375, 55.665035247802734, 55.677406311035156, 0.528403639793396)
Wait: 0
Wait: 47
(6.924513816833496, 7.3604655265808105, 7.391822814941406, 7.407444000244141, 47.19111633300781, 54.984493255615234, 55.257694244384766, 55.279

(7.594231605529785, 8.061213493347168, 8.086430549621582, 8.098993301391602, 51.783451080322266, 60.31538772583008, 60.60374069213867, 60.6256103515625, 60.6390266418457, 0.5061603784561157)
Wait: 0
Wait: 47
(7.5513739585876465, 8.015002250671387, 8.039484024047852, 8.051677703857422, 51.4930305480957, 59.975833892822266, 60.2618293762207, 60.283470153808594, 60.29679489135742, 0.5045285224914551)
Wait: 0
Wait: 47
(7.404250621795654, 7.858008861541748, 7.8813252449035645, 7.892937660217285, 50.49191665649414, 58.80830383300781, 59.087886810302734, 59.10897445678711, 59.12203598022461, 0.5084272623062134)
Wait: 0
Wait: 47
(7.292989730834961, 7.738205909729004, 7.75975227355957, 7.770480155944824, 49.73755645751953, 57.92661666870117, 58.20025634765625, 58.22077178955078, 58.23362350463867, 0.5070607662200928)
Wait: 0
Wait: 47
(7.21854829788208, 7.6580705642700195, 7.678447246551514, 7.688597202301025, 49.2327766418457, 57.33668518066406, 57.60636520385742, 57.626495361328125, 57.6392021

Wait: 0
Wait: 47
(6.848315715789795, 7.2776594161987305, 7.307179927825928, 7.3218841552734375, 46.67639923095703, 54.38154602050781, 54.649906158447266, 54.67086410522461, 54.68303298950195, 0.5179413557052612)
Wait: 0
Wait: 47
(6.854129314422607, 7.280032634735107, 7.306443691253662, 7.319598197937012, 46.725643157958984, 54.4321403503418, 54.69688034057617, 54.71728515625, 54.72943115234375, 0.5099533796310425)
Wait: 0
Wait: 47
(2109.982666015625, 2434.70751953125, 2602.431396484375, 2685.907470703125, 13894.291015625, 16531.0625, 16808.505859375, 16843.880859375, 16849.392578125, 1.016880989074707)
Wait: 0
Wait: 48
(1984.537109375, 2436.31494140625, 2714.708251953125, 2853.259765625, 12697.9931640625, 15377.8623046875, 15786.93359375, 15842.1962890625, 15848.72265625, 0.6833178997039795)
Wait: 0
Wait: 49
(825.3347778320312, 999.0512084960938, 1103.150390625, 1154.958251953125, 5316.7216796875, 6411.88330078125, 6567.66796875, 6588.521484375, 6591.1064453125, 0.5681109428405762)
Wai

...

Wait: 0
Wait: 47
(6.006911754608154, 6.352085590362549, 6.3520827293396, 6.35208797454834, 41.021080017089844, 47.73664474487305, 47.94023513793945, 47.95389938354492, 47.96428680419922, 0.48804664611816406)
Wait: 0
Wait: 47
(6.203203201293945, 6.559656143188477, 6.559653282165527, 6.559658527374268, 42.361549377441406, 49.29656219482422, 49.506805419921875, 49.52091598510742, 49.53164291381836, 0.4820427894592285)
Wait: 0
Wait: 48
(6.450491905212402, 6.821155071258545, 6.821152210235596, 6.821157932281494, 44.0502815246582, 51.261756896972656, 51.48038101196289, 51.49505615234375, 51.50621032714844, 0.4849752187728882)
Wait: 0
Wait: 47
(6.462244987487793, 6.833582878112793, 6.833580017089844, 6.833585739135742, 44.13054275512695, 51.35515594482422, 51.57417678833008, 51.5888786315918, 51.60005187988281, 0.49062514305114746)
Wait: 0
Wait: 47
(6.466370105743408, 6.837945461273193, 6.837942600250244, 6.837948322296143, 44.158714294433594, 51.387939453125, 51.60710144042969, 51.6218109130

Wait: 0
Wait: 47
(6.554419994354248, 6.9373459815979, 6.942529678344727, 6.945117473602295, 44.74409103393555, 52.08034896850586, 52.308860778808594, 52.324710845947266, 52.33610916137695, 0.5122098922729492)
Wait: 0
Wait: 47
(6.709740161895752, 7.102144718170166, 7.107787132263184, 7.110600471496582, 45.80336380004883, 53.31401824951172, 53.54835891723633, 53.56464767456055, 53.57631301879883, 0.5056061744689941)
Wait: 0
Wait: 48
(6.625821113586426, 7.0135297775268555, 7.019275665283203, 7.022141456604004, 45.2299690246582, 52.64698028564453, 52.878597259521484, 52.894718170166016, 52.9062385559082, 0.5020310878753662)
Wait: 0
Wait: 47
(6.526411056518555, 6.907981872558594, 6.913375377655029, 6.9160661697387695, 44.55216598510742, 51.85745620727539, 52.08528137207031, 52.101112365722656, 52.11245346069336, 0.5008183717727661)
Wait: 0
Wait: 47
(6.574355125427246, 6.958470344543457, 6.96368408203125, 6.9662909507751465, 44.8801155090332, 52.23871612548828, 52.46794891357422, 52.48385620

(6.827080249786377, 7.2432451248168945, 7.262908458709717, 7.272706031799316, 46.56163787841797, 54.22671127319336, 54.48225402832031, 54.50136184692383, 54.51338577270508, 0.5165442228317261)
Wait: 0
Wait: 48
(6.625522613525391, 7.029544830322266, 7.048749923706055, 7.058316230773926, 45.186614990234375, 52.62559127807617, 52.87372970581055, 52.892303466796875, 52.90397262573242, 0.5182981491088867)
Wait: 0
Wait: 49
(6.48423957824707, 6.880241394042969, 6.899528980255127, 6.909130096435547, 44.22154998779297, 51.50271224975586, 51.746158599853516, 51.76441955566406, 51.77585220336914, 0.5234802961349487)
Wait: 0
Wait: 47
(6.258554458618164, 6.640814781188965, 6.659458637237549, 6.668747425079346, 42.68230056762695, 49.71009826660156, 49.94511413574219, 49.96274948120117, 49.973777770996094, 0.5227617025375366)
Wait: 0
Wait: 47
(6.137035846710205, 6.511411666870117, 6.529316425323486, 6.538236618041992, 41.854736328125, 48.74544906616211, 48.975433349609375, 48.99265670776367, 49.00346

Wait: 0
Wait: 47
(6.122269153594971, 6.49305534362793, 6.508699893951416, 6.5164923667907715, 41.76082992553711, 48.63128662109375, 48.85799789428711, 48.87477493286133, 48.88554000854492, 0.5044132471084595)
Wait: 0
Wait: 47
(6.020599365234375, 6.384408950805664, 6.399119853973389, 6.406445026397705, 41.06940460205078, 47.824649810791016, 48.0467643737793, 48.063140869140625, 48.07371520996094, 0.5010112524032593)
Wait: 0
Wait: 48
(5.873589992523193, 6.227784633636475, 6.241536617279053, 6.24838399887085, 40.06842803955078, 46.6577262878418, 46.873680114746094, 46.88954544067383, 46.89985656738281, 0.5033879280090332)
Wait: 0
Wait: 47
(5.8342766761779785, 6.185540676116943, 6.198732376098633, 6.20530891418457, 39.801658630371094, 46.34608459472656, 46.560028076171875, 46.575706481933594, 46.585941314697266, 0.5075896978378296)
Wait: 0
Wait: 47
(5.91683292388916, 6.271284580230713, 6.283196449279785, 6.289134502410889, 40.36936569213867, 47.00396728515625, 47.2191276550293, 47.23476409

KeyboardInterrupt: 

In [7]:
p.stop()